Zero shot classification

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:


import os, sys, json, subprocess
from pathlib import Path

REPO = Path("/content/midterm-ad-topic")
if not REPO.exists():
    subprocess.run(["git", "clone", "https://github.com/rushi162004/midterm-ad-topic.git", str(REPO)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "pull"], check=True)
sys.path.insert(0, str(REPO / "src"))

DATA = Path("/content/drive/MyDrive/midterm-ad-topic/data")
OUT = DATA / "zero_shot"
OUT.mkdir(parents=True, exist_ok=True)

import pandas as pd, numpy as np, torch
msgs = pd.read_parquet(DATA / "meta_candidate_messages.parquet")
print(f"{len(msgs):,} unique messages | GPU available: {torch.cuda.is_available()}")
msgs.head(3)

In [ ]:
# TOPIC: what the ad is about. Each label has a description the model reads.
TOPICS = {
    "economy":       "the economy, jobs, wages, inflation or the cost of living",
    "healthcare":    "healthcare, health insurance, Medicare, Medicaid or prescription drugs",
    "immigration":   "immigration or the border",
    "crime":         "crime, policing or public safety",
    "abortion":      "abortion or reproductive rights",
    "education":     "schools, teachers or education",
    "taxes":         "taxes or government spending",
    "energy_env":    "energy, climate or the environment",
    "democracy":     "elections, voting rights or democracy",
    "guns":          "guns or gun policy",
    "social_security": "Social Security or retirement",
    "none_issue":    "no specific policy issue",
}

# PURPOSE: what the ad is trying to do
PURPOSES = {
    "promote":    "praising a candidate or their record",
    "attack":     "criticising or attacking an opponent",
    "contrast":   "comparing two candidates",
    "mobilise":   "urging people to vote or reminding them when and where to vote",
    "fundraise":  "asking for donations or money",
}

HYPOTHESIS = "This political ad is about {}."
PURPOSE_HYPOTHESIS = "This political ad is {}."

In [ ]:
TEST_N = 200
test = msgs.sample(TEST_N, random_state=42)[["msg_id", "payer", "body"]].copy()
test["true_topic"] = ""      # one key from TOPICS (the main topic)
test["true_purpose"] = ""    # one key from PURPOSES
test.to_csv(OUT / "test_set_to_label.csv", index=False)
print("Saved", len(test), "messages to label:", OUT / "test_set_to_label.csv")
print("Topic keys:", list(TOPICS))
print("Purpose keys:", list(PURPOSES))

In [ ]:
from transformers import pipeline

MODEL = "facebook/bart-large-mnli"
clf = pipeline("zero-shot-classification", model=MODEL,
               device=0 if torch.cuda.is_available() else -1,
               torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32)
print("Loaded", MODEL)

In [ ]:
def classify(texts, label_map, template, batch_size=16, max_chars=1000):
    """Return the best label and its score for each text, plus all label scores."""
    descs = list(label_map.values())
    back = {v: k for k, v in label_map.items()}
    texts = [t[:max_chars] for t in texts]          # very long ads get truncated anyway
    results = clf(texts, candidate_labels=descs, hypothesis_template=template,
                  multi_label=True, batch_size=batch_size)
    if isinstance(results, dict):
        results = [results]
    rows = []
    for r in results:
        scores = {back[l]: s for l, s in zip(r["labels"], r["scores"])}
        best = max(scores, key=scores.get)
        rows.append({"pred": best, "score": scores[best], **{f"s_{k}": v for k, v in scores.items()}})
    return pd.DataFrame(rows)

# quick smoke test on three messages
classify(msgs["body"].head(3).tolist(), TOPICS, HYPOTHESIS)

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

lab = pd.read_csv(OUT / "test_set_labelled.csv")
lab = lab[lab["true_topic"].notna() & (lab["true_topic"] != "")]
print(len(lab), "labelled messages")

topic_pred = classify(lab["body"].tolist(), TOPICS, HYPOTHESIS)
purpose_pred = classify(lab["body"].tolist(), PURPOSES, PURPOSE_HYPOTHESIS)
lab["pred_topic"], lab["topic_score"] = topic_pred["pred"].values, topic_pred["score"].values
lab["pred_purpose"] = purpose_pred["pred"].values

for name, true_col, pred_col in [("TOPIC", "true_topic", "pred_topic"),
                                 ("PURPOSE", "true_purpose", "pred_purpose")]:
    print(f"\n===== {name} =====")
    print(f"Accuracy: {accuracy_score(lab[true_col], lab[pred_col]):.2f} | "
          f"Macro-F1: {f1_score(lab[true_col], lab[pred_col], average='macro'):.2f}")
    print(classification_report(lab[true_col], lab[pred_col], zero_division=0))

labels_order = list(TOPICS)
cm = pd.DataFrame(confusion_matrix(lab["true_topic"], lab["pred_topic"], labels=labels_order),
                  index=labels_order, columns=labels_order)
cm.to_csv(OUT / "topic_confusion_matrix.csv")
lab.to_csv(OUT / "test_set_with_predictions.csv", index=False)
cm

In [ ]:
errors = lab[lab["true_topic"] != lab["pred_topic"]]
print(f"{len(errors)} topic errors out of {len(lab)}\n")
for _, r in errors.head(15).iterrows():
    print(f"TRUE: {r.true_topic:<15} PRED: {r.pred_topic:<15} score {r.topic_score:.2f}")
    print("   ", r.body[:250].replace("\n", " "), "\n")

In [ ]:
CHUNK = 500
parts = []
for start in range(0, len(msgs), CHUNK):
    path = OUT / f"pred_{start:06d}.parquet"
    if path.exists():
        parts.append(pd.read_parquet(path))
        continue
    chunk = msgs.iloc[start:start + CHUNK]
    t = classify(chunk["body"].tolist(), TOPICS, HYPOTHESIS).add_prefix("topic_")
    p = classify(chunk["body"].tolist(), PURPOSES, PURPOSE_HYPOTHESIS).add_prefix("purpose_")
    res = pd.concat([chunk[["msg_id"]].reset_index(drop=True), t, p], axis=1)
    res.to_parquet(path, index=False)
    parts.append(res)
    print(f"{min(start + CHUNK, len(msgs)):,} / {len(msgs):,} done")

preds = pd.concat(parts, ignore_index=True)
labelled = msgs.merge(preds, on="msg_id", how="left")
labelled.to_parquet(OUT / "messages_labelled.parquet", index=False)
print("Saved", len(labelled), "labelled messages")

In [ ]:
by_msgs = labelled["topic_pred"].value_counts(normalize=True).rename("share_of_messages")
by_spend = (labelled.groupby("topic_pred")["spend_mid"].sum()
            / labelled["spend_mid"].sum()).rename("share_of_spend")
summary = pd.concat([by_msgs, by_spend], axis=1).sort_values("share_of_spend", ascending=False).round(3)
summary